# Subject-Independent BCI Competition IV-2a: Hybrid PLV-GAT EEG Classifier

This notebook upgrades the original `gat-plv-with-ex-bci(1).ipynb` into a **true subject-independent (LOSO) 4-class motor-imagery pipeline**.

### Main changes

- **LOSO (Leave-One-Subject-Out)**: one subject is completely held out as the test subject.
- Uses all **4 BCI IV-2a classes** by default: left hand, right hand, feet, tongue.
- Correct **22-node EEG graph** construction.
- **Multiband PLV edge features** instead of an incorrect 1000×1000 time-sample graph.
- **Multiband log-relative-power node features**.
- **Hybrid architecture**:
  - temporal CNN branch for time-domain MI dynamics
  - PLV-weighted GATv2 branch for functional connectivity
  - gated fusion + classifier
- **Subject-wise validation** for early stopping; the held-out test subject is never used to choose an epoch.
- Per-subject normalization and training-only feature scaling.
- Top-k PLV sparsification to reduce noisy fully-connected graphs.
- Caching of precomputed features so later LOSO folds do not repeatedly recompute PLV.
- Optional **binary left-vs-right mode** through one configuration flag.
- Saves fold-level results, confusion matrices, and the final summary.

> The goal is **70%+ mean LOSO accuracy**, not a guaranteed score. Actual accuracy depends on the exact `.mat` files, preprocessing, random seed, hardware, and training time.


## Dataset assumptions

BCI Competition IV-2a contains 9 subjects, 22 EEG channels, 250 Hz sampling, and four motor-imagery classes. Each subject has one labeled training session and one evaluation session. The original competition evaluation labels were withheld; for a reproducible **subject-independent research experiment**, this notebook therefore uses the labeled `A0XT.mat` data and treats one subject's entire labeled training set as the held-out LOSO test set.

The epoch window defaults to **2.0–6.0 s after trial onset**, matching the cue/imagery timing described in the official dataset description.

Official dataset description: https://www.bbci.de/competition/iv/desc_2a.pdf


In [ ]:
# ============================================================
# 1. INSTALL / IMPORTS
# ============================================================
# PyTorch itself should be installed in your Jupyter environment.
# If PyTorch Geometric is missing, install it from inside the notebook.
try:
    import torch_geometric  # noqa: F401
except ImportError:
    get_ipython().run_line_magic('pip', 'install -q torch-geometric')

import os, glob, json, math, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.io as sio
from scipy import signal as sig
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, balanced_accuracy_score, cohen_kappa_score, confusion_matrix, classification_report
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import WeightedRandomSampler
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATv2Conv, GraphNorm, global_mean_pool, global_max_pool
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# ============================================================
# 2. CONFIGURATION
# ============================================================
# >>> SET THIS TO THE FOLDER CONTAINING A01T.mat ... A09T.mat <<<
# Windows: r"C:\datasets\BCICIV2a"
# macOS:   "/Users/yourname/datasets/BCICIV2a"
# Linux:   "/home/yourname/datasets/BCICIV2a"
DATA_ROOT = os.environ.get('BCI_2A_ROOT', '').strip()

if not DATA_ROOT:
    candidates = [os.getcwd(), os.path.join(os.getcwd(), 'BCI_IV_2a'), os.path.join(os.getcwd(), 'BCICIV2a'), '/kaggle/input/bci-competition-iv-data-sets-2a', '/content']
    for c in candidates:
        if os.path.isdir(c) and glob.glob(os.path.join(c, '**', 'A01T.mat'), recursive=True):
            DATA_ROOT = c; break
print('DATA_ROOT =', DATA_ROOT)

BINARY_LR = False
ALL_SUBJECTS = list(range(1, 10))
if BINARY_LR:
    CLASS_IDS = [1, 2]; CLASS_NAMES = ['Left hand', 'Right hand']
else:
    CLASS_IDS = [1, 2, 3, 4]; CLASS_NAMES = ['Left hand', 'Right hand', 'Both feet', 'Tongue']

FS = 250; N_EEG = 22
EPOCH_START_SEC = 2.0; EPOCH_STOP_SEC = 6.0
BROAD_BAND = (8.0, 30.0)
BANDS = [(8.0, 12.0), (12.0, 16.0), (16.0, 22.0), (22.0, 30.0)]
BAND_NAMES = ['8-12', '12-16', '16-22', '22-30']
TOP_K = 6; PLV_EPS = 1e-6

BATCH_SIZE = 64; MAX_EPOCHS = 100; PATIENCE = 18
LR = 2e-3; WEIGHT_DECAY = 1e-4; DROPOUT = 0.25
LABEL_SMOOTHING = 0.05; GRAD_CLIP = 2.0
VAL_FRACTION = 0.20; QUICK_RUN = False
FORCE_RECOMPUTE_CACHE = False
# Optional: unsupervised per-subject Euclidean alignment. This uses the
# unlabeled distribution of each subject, including the held-out target, but
# never uses target labels. Keep False for strict inductive LOSO.
USE_UNLABELED_SUBJECT_ALIGNMENT = False
if QUICK_RUN: MAX_EPOCHS = 20; PATIENCE = 6

CACHE_DIR = os.path.join(os.path.abspath(DATA_ROOT or os.getcwd()), '_plv_gat_cache')
RESULTS_DIR = os.path.join(os.path.abspath(DATA_ROOT or os.getcwd()), '_plv_gat_results')
os.makedirs(CACHE_DIR, exist_ok=True); os.makedirs(RESULTS_DIR, exist_ok=True)
USE_CAR = True; USE_EOG_REGRESSION_IF_AVAILABLE = True

print('Classes:', dict(zip(CLASS_IDS, CLASS_NAMES)))
print('Epoch:', EPOCH_START_SEC, 'to', EPOCH_STOP_SEC, 's')
print('Bands:', list(zip(BAND_NAMES, BANDS)))
print('TOP_K:', TOP_K)


In [ ]:
# ============================================================
# 3. DATA DISCOVERY / VALIDATION
# ============================================================
def find_subject_file(subject_id, suffix='T'):
    if not DATA_ROOT: return None
    matches = glob.glob(os.path.join(DATA_ROOT, '**', f'A{subject_id:02d}{suffix}.mat'), recursive=True)
    return matches[0] if matches else None

subject_files = {sid: find_subject_file(sid, 'T') for sid in ALL_SUBJECTS}
for sid, path in subject_files.items():
    print(f'A{sid:02d}T.mat -> {path}')
missing = [sid for sid, p in subject_files.items() if p is None]
if missing:
    msg = 'Missing training files: ' + ', '.join(f'A{s:02d}T.mat' for s in missing)
    raise FileNotFoundError(msg + '\nSet DATA_ROOT to the folder containing the .mat files.')


In [ ]:
# ============================================================
# 4. ROBUST MATLAB STRUCT HELPERS + BCI 2a LOADER
# ============================================================
def _as_list(obj):
    return list(obj.ravel()) if isinstance(obj, np.ndarray) else [obj]

def _get_field(obj, name, default=None):
    if obj is None: return default
    if hasattr(obj, name): return getattr(obj, name)
    if isinstance(obj, dict): return obj.get(name, default)
    return default

def _fix_eeg_orientation(x):
    x = np.asarray(x, dtype=np.float64)
    if x.ndim != 2: raise ValueError(f'Expected 2-D EEG array, got {x.shape}')
    if x.shape[1] >= 22: return x
    if x.shape[0] >= 22 and x.shape[1] < 22: return x.T
    raise ValueError(f'Cannot identify EEG channel dimension from {x.shape}')

def _maybe_eog_regress(eeg, eog):
    if eog is None or eog.shape[1] == 0: return eeg
    valid = np.isfinite(eog).all(axis=1) & np.isfinite(eeg).all(axis=1)
    if valid.sum() < max(100, 10 * eog.shape[1]): return eeg
    E, Y = eog[valid], eeg[valid]
    A = np.column_stack([np.ones(len(E)), E])
    beta, *_ = np.linalg.lstsq(A, Y, rcond=None)
    corrected = eeg.copy()
    corrected_valid = np.isfinite(eog).all(axis=1) & np.isfinite(eeg).all(axis=1)
    A_all = np.column_stack([np.ones(corrected_valid.sum()), eog[corrected_valid]])
    corrected[corrected_valid] = eeg[corrected_valid] - A_all @ beta
    return corrected

def _clean_trial(trial):
    trial = np.nan_to_num(np.asarray(trial, dtype=np.float64), nan=0.0, posinf=0.0, neginf=0.0)
    if USE_CAR: trial = trial - np.mean(trial, axis=1, keepdims=True)
    med = np.median(trial, axis=0, keepdims=True)
    mad = np.median(np.abs(trial - med), axis=0, keepdims=True)
    trial = (trial - med) / (1.4826 * mad + 1e-6)
    return trial.astype(np.float32)

def euclidean_align_trials(trials, reg=1e-4):
    """Unsupervised Euclidean Alignment (EA) within one subject.

    trials: [N, C, T]. No class labels are used.
    """
    X = trials.astype(np.float64, copy=False)
    C = X.shape[1]
    covs = []
    eye = np.eye(C)
    for trial in X:
        trial0 = trial - trial.mean(axis=1, keepdims=True)
        cov = (trial0 @ trial0.T) / max(1, trial0.shape[1] - 1)
        cov = cov + reg * (np.trace(cov) / C + 1e-6) * eye
        covs.append(cov)
    R = np.mean(covs, axis=0)
    evals, evecs = np.linalg.eigh(R)
    evals = np.maximum(evals, 1e-10)
    W = (evecs * (1.0 / np.sqrt(evals))) @ evecs.T
    aligned = np.einsum('ij,njt->nit', W, X)
    return aligned.astype(np.float32)

def load_bci2a_subject(subject_id):
    path = subject_files[subject_id]
    mat = sio.loadmat(path, squeeze_me=True, struct_as_record=False)
    if 'data' not in mat: raise KeyError(f"'data' variable not found in {path}")
    sessions = _as_list(mat['data'])
    crop_start = int(round(EPOCH_START_SEC * FS)); crop_stop = int(round(EPOCH_STOP_SEC * FS))
    trials, labels = [], []

    for sess in sessions:
        X = _get_field(sess, 'X'); trial_pos = _get_field(sess, 'trial'); y = _get_field(sess, 'y')
        if X is None or trial_pos is None or y is None: continue
        X = _fix_eeg_orientation(X)
        trial_pos = np.asarray(trial_pos).reshape(-1); y = np.asarray(y).reshape(-1)
        for p, lab in zip(trial_pos[:len(y)], y):
            lab = int(np.round(float(lab)))
            if lab not in CLASS_IDS: continue
            start = int(np.round(float(p))); stop = start + crop_stop
            if start < 0: continue
            if stop > len(X):
                start2 = max(0, start - 1); stop2 = start2 + crop_stop
                if stop2 > len(X): continue
                start, stop = start2, stop2
            segment = X[start:stop]
            if segment.shape[0] < crop_stop: continue
            eeg = segment[:, :N_EEG]
            eog = segment[:, N_EEG:25] if segment.shape[1] >= 25 else None
            if USE_EOG_REGRESSION_IF_AVAILABLE and eog is not None: eeg = _maybe_eog_regress(eeg, eog)
            eeg = eeg[crop_start:crop_stop]
            if eeg.shape[0] != crop_stop - crop_start: continue
            trials.append(_clean_trial(eeg).T); labels.append(CLASS_IDS.index(lab))

    if not trials: raise RuntimeError(f'No labeled epochs extracted from {path}')
    return np.stack(trials).astype(np.float32), np.asarray(labels, dtype=np.int64)

subject_raw = {}
for sid in ALL_SUBJECTS:
    Xs, ys = load_bci2a_subject(sid); subject_raw[sid] = (Xs, ys)
    print(f'A{sid:02d}: X={Xs.shape}, class_counts={np.bincount(ys, minlength=len(CLASS_IDS)).tolist()}')


In [ ]:
# ============================================================
# 5. FEATURE EXTRACTION
# ============================================================
def bandpass_sos(x, low, high, fs=FS, order=4):
    sos = sig.butter(order, [low, high], btype='bandpass', fs=fs, output='sos')
    return sig.sosfiltfilt(sos, x, axis=-1).astype(np.float32)

def log_bandpower(x, low, high, fs=FS):
    nper = min(int(2*fs), x.shape[-1])
    f, pxx = sig.welch(x, fs=fs, nperseg=nper, axis=-1)
    mask = (f >= low) & (f <= high)
    power = np.trapz(pxx[..., mask], f[mask], axis=-1)
    return np.log10(power + 1e-8).astype(np.float32)

def phase_locking_matrix(x):
    analytic = sig.hilbert(x, axis=-1); phase = np.angle(analytic); z = np.exp(1j*phase)
    plv = np.abs(z @ np.conj(z).T) / x.shape[-1]
    np.fill_diagonal(plv, 0.0)
    return plv.astype(np.float32)

def topk_graph_from_plv(plv_bands, top_k=TOP_K):
    mean_plv = np.mean(plv_bands, axis=0); C = mean_plv.shape[0]
    src, dst, attrs = [], [], []
    for dst_node in range(C):
        scores = mean_plv[:, dst_node].copy(); scores[dst_node] = -np.inf
        nbrs = np.argpartition(scores, -top_k)[-top_k:]; nbrs = nbrs[np.argsort(scores[nbrs])[::-1]]
        for src_node in nbrs:
            src.append(int(src_node)); dst.append(int(dst_node)); attrs.append(plv_bands[:, src_node, dst_node])
    return np.asarray([src, dst], dtype=np.int64), np.asarray(attrs, dtype=np.float32)

def extract_subject_features(trials):
    n_trials, n_ch, _ = trials.shape; E = n_ch * TOP_K
    temporal_all = np.zeros_like(trials, dtype=np.float32)
    node_features_all = np.zeros((n_trials, n_ch, len(BANDS)+1), dtype=np.float32)
    edge_index_all = np.zeros((n_trials, 2, E), dtype=np.int64)
    edge_attr_all = np.zeros((n_trials, E, len(BANDS)), dtype=np.float32)

    for k in tqdm(range(n_trials), desc='Feature extraction', leave=False):
        trial = trials[k]; broad = bandpass_sos(trial, *BROAD_BAND); temporal_all[k] = broad
        band_logs, plv_bands = [], []
        for low, high in BANDS:
            xb = bandpass_sos(trial, low, high)
            band_logs.append(log_bandpower(xb, low, high)); plv_bands.append(phase_locking_matrix(xb))
        band_logs = np.stack(band_logs, axis=-1)
        rel = band_logs - np.mean(band_logs, axis=-1, keepdims=True)
        total_log = log_bandpower(broad, *BROAD_BAND)[:, None]
        node_features_all[k] = np.concatenate([rel, total_log], axis=-1)
        edge_index_all[k], edge_attr_all[k] = topk_graph_from_plv(np.stack(plv_bands, axis=0))
    return temporal_all, node_features_all, edge_index_all, edge_attr_all

def cache_path(subject_id):
    mode = 'binary' if BINARY_LR else '4class'
    return os.path.join(CACHE_DIR, f'A{subject_id:02d}_{mode}.npz')

def build_or_load_features(subject_id, force_recompute=FORCE_RECOMPUTE_CACHE):
    p = cache_path(subject_id)
    if os.path.exists(p) and not force_recompute:
        z = np.load(p)
        return z['temporal'], z['node_features'], z['edge_index'], z['edge_attr'], z['labels']
    trials, labels = subject_raw[subject_id]
    temporal, node_features, edge_index, edge_attr = extract_subject_features(trials)
    np.savez_compressed(p, temporal=temporal.astype(np.float16), node_features=node_features.astype(np.float32), edge_index=edge_index.astype(np.int64), edge_attr=edge_attr.astype(np.float16), labels=labels.astype(np.int64))
    return temporal, node_features, edge_index, edge_attr, labels

subject_features = {}
for sid in ALL_SUBJECTS:
    print(f'\nPreparing A{sid:02d}...')
    subject_features[sid] = build_or_load_features(sid)
    t,nf,ei,ea,y = subject_features[sid]
    print(f'  temporal={t.shape}, node_features={nf.shape}, edge_index={ei.shape}, edge_attr={ea.shape}')


## Why the graph is now correct

Each trial is represented as a graph with exactly **22 nodes**, one for each EEG electrode.

- **Node features:** 5 values per electrode: four relative log-power bands plus total 8–30 Hz log power.
- **Edges:** top-6 PLV connections for each destination node.
- **Edge features:** a 4-dimensional PLV vector containing the same four frequency bands.

The GAT therefore sees the actual EEG electrode graph and the actual PLV magnitudes.

In [ ]:
# ============================================================
# 6. TRAINING FEATURE SCALER
# ============================================================
def fit_node_scaler(subject_ids):
    chunks = [subject_features[sid][1].reshape(-1, subject_features[sid][1].shape[-1]) for sid in subject_ids]
    scaler = StandardScaler(); scaler.fit(np.concatenate(chunks, axis=0)); return scaler

def apply_node_scaler(node_features, scaler):
    shape = node_features.shape
    return scaler.transform(node_features.reshape(-1, shape[-1])).reshape(shape).astype(np.float32)


In [ ]:
# ============================================================
# 7. PYTORCH GEOMETRIC DATASET CONVERSION
# ============================================================
def subject_to_graphs(subject_id, scaler=None):
    temporal, node_feat, edge_index, edge_attr, labels = subject_features[subject_id]
    if scaler is not None: node_feat = apply_node_scaler(node_feat, scaler)
    graphs = []
    for i in range(len(labels)):
        graphs.append(Data(
            x=torch.tensor(node_feat[i], dtype=torch.float32),
            edge_index=torch.tensor(edge_index[i], dtype=torch.long),
            edge_attr=torch.tensor(edge_attr[i], dtype=torch.float32),
            temporal=torch.tensor(temporal[i], dtype=torch.float32),
            y=torch.tensor(labels[i], dtype=torch.long),
            subject=torch.tensor([subject_id], dtype=torch.long),
        ))
    return graphs

def make_balanced_sampler(graphs):
    labels = np.asarray([int(g.y.item()) for g in graphs])
    counts = np.bincount(labels, minlength=len(CLASS_IDS)).astype(np.float64); counts[counts==0]=1.0
    weights = (1.0/counts)[labels]
    return WeightedRandomSampler(torch.as_tensor(weights, dtype=torch.double), len(weights), replacement=True)

def make_loaders(train_graphs, val_graphs):
    train_loader = DataLoader(train_graphs, batch_size=BATCH_SIZE, sampler=make_balanced_sampler(train_graphs), num_workers=0, pin_memory=torch.cuda.is_available())
    val_loader = DataLoader(val_graphs, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())
    return train_loader, val_loader


## Hybrid model

The classifier uses two complementary branches:

1. **Temporal CNN:** learns time-domain motor-imagery dynamics from the 22-channel 8–30 Hz signal.
2. **PLV-GATv2:** learns spatial/functional patterns from the 22-node graph while receiving multiband PLV as `edge_attr`.
3. **Gated fusion:** learns how much to trust temporal versus connectivity information for each trial.

In [ ]:
# ============================================================
# 8. MODEL
# ============================================================
class TemporalBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, dropout=0.2):
        super().__init__(); pad=kernel_size//2
        self.net=nn.Sequential(
            nn.Conv1d(in_ch,out_ch,kernel_size,padding=pad,bias=False), nn.BatchNorm1d(out_ch), nn.GELU(),
            nn.Conv1d(out_ch,out_ch,15,padding=7,groups=out_ch,bias=False), nn.BatchNorm1d(out_ch), nn.GELU(),
            nn.Conv1d(out_ch,out_ch,1,bias=False), nn.BatchNorm1d(out_ch), nn.GELU(), nn.Dropout(dropout))
        self.skip=nn.Conv1d(in_ch,out_ch,1) if in_ch!=out_ch else nn.Identity()
    def forward(self,x): return self.net(x)+self.skip(x)

class TemporalEncoder(nn.Module):
    def __init__(self,n_channels=22,dropout=0.2):
        super().__init__(); self.b1=TemporalBlock(n_channels,32,31,dropout); self.p1=nn.AvgPool1d(4); self.b2=TemporalBlock(32,64,21,dropout); self.p2=nn.AvgPool1d(4); self.b3=TemporalBlock(64,96,15,dropout); self.p3=nn.AvgPool1d(2); self.h=nn.Sequential(nn.AdaptiveAvgPool1d(1),nn.Flatten(),nn.Linear(96,128),nn.GELU(),nn.Dropout(dropout))
    def forward(self,x): return self.h(self.p3(self.b3(self.p2(self.b2(self.p1(self.b1(x)))))))

class GraphEncoder(nn.Module):
    def __init__(self,in_dim=5,hidden=32,heads=4,edge_dim=4,dropout=0.2):
        super().__init__(); H=hidden*heads
        self.in_proj=nn.Linear(in_dim,H)
        self.g1=GATv2Conv(H,hidden,heads=heads,concat=True,dropout=dropout,edge_dim=edge_dim,add_self_loops=True)
        self.n1=GraphNorm(H)
        self.g2=GATv2Conv(H,hidden,heads=heads,concat=True,dropout=dropout,edge_dim=edge_dim,add_self_loops=True)
        self.n2=GraphNorm(H)
        self.out=nn.Sequential(nn.Linear(2*H,160),nn.GELU(),nn.Dropout(dropout),nn.Linear(160,128))
    def forward(self,x,edge_index,edge_attr,batch):
        h0=self.in_proj(x); h=self.g1(x,edge_index,edge_attr=edge_attr); h=F.gelu(self.n1(h,batch)+h0)
        r=h; h=self.g2(h,edge_index,edge_attr=edge_attr); h=F.gelu(self.n2(h,batch)+r)
        return self.out(torch.cat([global_mean_pool(h,batch),global_max_pool(h,batch)],dim=-1))

class HybridPLVGAT(nn.Module):
    def __init__(self,n_classes,dropout=DROPOUT):
        super().__init__(); self.temporal=TemporalEncoder(N_EEG,dropout); self.graph=GraphEncoder(len(BANDS)+1,32,4,len(BANDS),dropout)
        self.tproj=nn.Sequential(nn.Linear(128,128),nn.LayerNorm(128),nn.GELU()); self.gproj=nn.Sequential(nn.Linear(128,128),nn.LayerNorm(128),nn.GELU())
        self.gate=nn.Sequential(nn.Linear(256,128),nn.GELU(),nn.Linear(128,1),nn.Sigmoid())
        self.cls=nn.Sequential(nn.Linear(128,128),nn.LayerNorm(128),nn.GELU(),nn.Dropout(dropout),nn.Linear(128,n_classes))
    def forward(self,data,return_embedding=False):
        t=self.tproj(self.temporal(data.temporal)); g=self.gproj(self.graph(data.x,data.edge_index,data.edge_attr,data.batch)); gate=self.gate(torch.cat([g,t],dim=-1)); fused=gate*g+(1-gate)*t; logits=self.cls(fused)
        return (logits,fused,gate) if return_embedding else logits

# Smoke test
_scaler=fit_node_scaler([1]); _g=subject_to_graphs(1,_scaler)[0]; _b=next(iter(DataLoader([_g],batch_size=1))).to(device); _m=HybridPLVGAT(len(CLASS_IDS)).to(device)
with torch.no_grad(): print('Smoke-test logits shape:', tuple(_m(_b).shape))


In [ ]:
# ============================================================
# 9. TRAINING / EVALUATION
# ============================================================
def class_weights_from_graphs(graphs):
    y=np.asarray([int(g.y.item()) for g in graphs]); c=np.bincount(y,minlength=len(CLASS_IDS)).astype(np.float32); c[c==0]=1.0
    return torch.tensor(c.sum()/(len(CLASS_IDS)*c),dtype=torch.float32,device=device)

def train_one_epoch(model,loader,optimizer,criterion):
    model.train(); losses=[]; yt=[]; yp=[]
    for batch in loader:
        batch=batch.to(device); optimizer.zero_grad(set_to_none=True); logits=model(batch); loss=criterion(logits,batch.y); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),GRAD_CLIP); optimizer.step(); losses.append(loss.item()); yt.extend(batch.y.detach().cpu().numpy()); yp.extend(logits.argmax(1).detach().cpu().numpy())
    return float(np.mean(losses)), float(accuracy_score(yt,yp))

@torch.no_grad()
def evaluate(model,loader):
    model.eval(); yt=[]; yp=[]; probs=[]; losses=[]
    for batch in loader:
        batch=batch.to(device); logits=model(batch); losses.append(F.cross_entropy(logits,batch.y).item()); p=torch.softmax(logits,1); yt.extend(batch.y.cpu().numpy()); yp.extend(logits.argmax(1).cpu().numpy()); probs.append(p.cpu().numpy())
    yt=np.asarray(yt); yp=np.asarray(yp); probs=np.concatenate(probs,axis=0)
    return {'loss':float(np.mean(losses)),'accuracy':float(accuracy_score(yt,yp)),'balanced_accuracy':float(balanced_accuracy_score(yt,yp)),'kappa':float(cohen_kappa_score(yt,yp)),'y_true':yt,'y_pred':yp,'probs':probs}

def train_model(train_graphs,val_graphs,fold_name='fold'):
    train_loader,val_loader=make_loaders(train_graphs,val_graphs); model=HybridPLVGAT(len(CLASS_IDS)).to(device)
    criterion=nn.CrossEntropyLoss(weight=class_weights_from_graphs(train_graphs),label_smoothing=LABEL_SMOOTHING)
    optimizer=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
    scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=MAX_EPOCHS,eta_min=LR*0.05)
    best=-np.inf; best_epoch=0; patience=0; best_state=None; history=[]
    for epoch in range(1,MAX_EPOCHS+1):
        tr_loss,tr_acc=train_one_epoch(model,train_loader,optimizer,criterion); val=evaluate(model,val_loader); scheduler.step()
        history.append({'epoch':epoch,'train_loss':tr_loss,'train_accuracy':tr_acc,'val_loss':val['loss'],'val_accuracy':val['accuracy'],'val_balanced_accuracy':val['balanced_accuracy'],'val_kappa':val['kappa'],'lr':optimizer.param_groups[0]['lr']})
        if val['accuracy']>best+1e-4:
            best=val['accuracy']; best_epoch=epoch; patience=0; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        else: patience+=1
        if epoch==1 or epoch%5==0: print(f'{fold_name} | ep {epoch:03d} | train={tr_acc:.3f} | val={val["accuracy"]:.3f} | kappa={val["kappa"]:.3f}')
        if patience>=PATIENCE: print(f'{fold_name} | early stop at epoch {epoch}'); break
    model.load_state_dict(best_state); model.to(device); return model,pd.DataFrame(history),best_epoch,float(best)


In [ ]:
# ============================================================
# 10. TRUE LOSO SPLIT
# ============================================================
def choose_source_validation_subjects(target_subject, source_subjects, seed=SEED):
    src=np.asarray(source_subjects,dtype=int); splitter=GroupShuffleSplit(n_splits=1,test_size=max(1,int(round(len(src)*VAL_FRACTION))),random_state=seed+target_subject); idx=np.arange(len(src)); tr,va=next(splitter.split(idx,groups=src)); train_subjects=src[tr].tolist(); val_subjects=src[va].tolist(); return sorted(train_subjects), sorted(val_subjects)

def flatten_subject_graphs(subject_ids,scaler):
    out=[]
    for sid in subject_ids: out.extend(subject_to_graphs(sid,scaler=scaler))
    return out


In [ ]:
# ============================================================
# 11. RUN FULL SUBJECT-INDEPENDENT LOSO
# ============================================================
all_fold_results=[]; all_test_predictions={}; all_histories={}

for target_subject in ALL_SUBJECTS:
    print('\n'+'='*80); print(f'LOSO TARGET SUBJECT: A{target_subject:02d}'); print('='*80)
    source_subjects=[s for s in ALL_SUBJECTS if s!=target_subject]
    train_subjects,val_subjects=choose_source_validation_subjects(target_subject,source_subjects)
    print('Source train:',[f'A{s:02d}' for s in train_subjects]); print('Validation:',[f'A{s:02d}' for s in val_subjects]); print('Held-out test:',f'A{target_subject:02d}')
    scaler=fit_node_scaler(train_subjects)
    train_graphs=flatten_subject_graphs(train_subjects,scaler); val_graphs=flatten_subject_graphs(val_subjects,scaler); test_graphs=flatten_subject_graphs([target_subject],scaler)
    model,history_df,best_epoch,best_val_acc=train_model(train_graphs,val_graphs,fold_name=f'A{target_subject:02d}')
    test_loader=DataLoader(test_graphs,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())
    test=evaluate(model,test_loader); cm=confusion_matrix(test['y_true'],test['y_pred'],labels=list(range(len(CLASS_IDS))))
    rec={'target_subject':target_subject,'train_subjects':train_subjects,'val_subjects':val_subjects,'best_epoch':int(best_epoch),'best_val_accuracy':float(best_val_acc),'test_accuracy':test['accuracy'],'test_balanced_accuracy':test['balanced_accuracy'],'test_kappa':test['kappa'],'confusion_matrix':cm.tolist()}
    all_fold_results.append(rec); all_test_predictions[target_subject]={'y_true':test['y_true'].tolist(),'y_pred':test['y_pred'].tolist(),'probs':test['probs'].tolist()}; all_histories[target_subject]=history_df.to_dict('records')
    print(f'A{target_subject:02d} TEST -> accuracy={test["accuracy"]:.4f}, balanced_accuracy={test["balanced_accuracy"]:.4f}, kappa={test["kappa"]:.4f}')
    torch.save(model.state_dict(), os.path.join(RESULTS_DIR, f'best_A{target_subject:02d}.pt'))
    history_df.to_csv(os.path.join(RESULTS_DIR, f'history_A{target_subject:02d}.csv'), index=False)
    del model,train_graphs,val_graphs,test_graphs
    if torch.cuda.is_available(): torch.cuda.empty_cache()


In [ ]:
# ============================================================
# 12. FINAL LOSO SUMMARY
# ============================================================
results_df=pd.DataFrame(all_fold_results)
show=results_df[['target_subject','best_epoch','test_accuracy','test_balanced_accuracy','test_kappa']].copy(); show['subject']=show.pop('target_subject').map(lambda x:f'A{x:02d}'); show=show[['subject','best_epoch','test_accuracy','test_balanced_accuracy','test_kappa']]
display(show)
summary={'mean_accuracy':float(results_df.test_accuracy.mean()),'std_accuracy':float(results_df.test_accuracy.std(ddof=1)),'mean_balanced_accuracy':float(results_df.test_balanced_accuracy.mean()),'std_balanced_accuracy':float(results_df.test_balanced_accuracy.std(ddof=1)),'mean_kappa':float(results_df.test_kappa.mean()),'std_kappa':float(results_df.test_kappa.std(ddof=1))}
print('\n'+'='*80); print('FINAL SUBJECT-INDEPENDENT RESULT'); print('='*80); print(f"Mean LOSO Accuracy      : {summary['mean_accuracy']:.4f}"); print(f"Std LOSO Accuracy       : {summary['std_accuracy']:.4f}"); print(f"Mean Balanced Accuracy  : {summary['mean_balanced_accuracy']:.4f}"); print(f"Mean Cohen Kappa        : {summary['mean_kappa']:.4f}")
if summary['mean_accuracy']>=0.70: print('\n✅ TARGET REACHED: mean LOSO accuracy >= 70%')
else: print('\n⚠️ TARGET NOT YET REACHED: mean LOSO accuracy < 70%; use tuning below without changing LOSO protocol.')


In [ ]:
# ============================================================
# 13. CONFUSION MATRICES + PERFORMANCE PLOT
# ============================================================
fig,axes=plt.subplots(3,3,figsize=(13,12)); axes=axes.ravel()
for idx,sid in enumerate(ALL_SUBJECTS):
    rec=next(r for r in all_fold_results if r['target_subject']==sid); cm=np.asarray(rec['confusion_matrix']); ax=axes[idx]
    sns.heatmap(cm,annot=True,fmt='d',cbar=False,ax=ax,xticklabels=CLASS_NAMES,yticklabels=CLASS_NAMES,cmap='Blues')
    ax.set_title(f'A{sid:02d} | Acc={rec["test_accuracy"]:.3f}'); ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.tight_layout(); plt.show()

plt.figure(figsize=(12,5)); x=np.arange(len(results_df)); plt.bar(x,results_df['test_accuracy'].values); plt.axhline(.70,linestyle='--',linewidth=2,label='70% target'); plt.xticks(x,[f'A{s:02d}' for s in results_df['target_subject']]); plt.ylabel('LOSO accuracy'); plt.ylim(0,1); plt.title('Subject-independent accuracy by held-out subject'); plt.legend(); plt.grid(axis='y',alpha=.25); plt.show()


In [ ]:
# ============================================================
# 14. SAVE PAPER-READY RESULTS
# ============================================================
results_path=os.path.join(RESULTS_DIR,'loso_subject_results.csv'); json_path=os.path.join(RESULTS_DIR,'loso_subject_results.json'); pred_path=os.path.join(RESULTS_DIR,'loso_predictions.json'); config_path=os.path.join(RESULTS_DIR,'experiment_config.json')
results_df.to_csv(results_path,index=False)
with open(json_path,'w') as f: json.dump({'summary':summary,'folds':all_fold_results},f,indent=2)
with open(pred_path,'w') as f: json.dump(all_test_predictions,f)
config={'task':'BCI Competition IV-2a subject-independent LOSO','binary_lr':BINARY_LR,'classes':CLASS_NAMES,'subjects':ALL_SUBJECTS,'fs':FS,'n_eeg':N_EEG,'epoch_start_sec':EPOCH_START_SEC,'epoch_stop_sec':EPOCH_STOP_SEC,'broad_band':BROAD_BAND,'bands':BANDS,'top_k':TOP_K,'batch_size':BATCH_SIZE,'max_epochs':MAX_EPOCHS,'patience':PATIENCE,'lr':LR,'weight_decay':WEIGHT_DECAY,'seed':SEED,'device':str(device),'use_unlabeled_subject_alignment':USE_UNLABELED_SUBJECT_ALIGNMENT}
with open(config_path,'w') as f: json.dump(config,f,indent=2)
print('Saved:',results_path,json_path,pred_path,config_path,sep='\n')


# 15. TUNING ORDER IF THE MEAN IS BELOW 70%

Do **not** alter the LOSO test protocol to make the score higher.

1. Increase `MAX_EPOCHS` to 120–160 and `PATIENCE` to 20–25.
2. Test `TOP_K = 4, 6, 8`.
3. Test epoch windows `2.0–6.0`, `2.5–6.0`, and `3.0–6.0` s.
4. Test `LR = 1e-3, 1.5e-3, 2e-3`.
5. Test `DROPOUT = 0.15, 0.25, 0.35`.
6. For a final study, repeat the full LOSO run with 3 random seeds.

For a binary experiment, set `BINARY_LR = True` and rerun from the configuration cell.

# 16. PAPER-QUALITY REPORTING CHECKLIST

Report accuracy for A01–A09, mean ± standard deviation across the 9 held-out subjects, balanced accuracy, Cohen's kappa, the aggregate confusion matrix, exact epoch/frequency settings, LOSO split definition, random seeds, and comparison with at least one baseline.

This notebook does **not** use the held-out subject labels for early stopping, hyperparameter selection, or checkpoint selection.